In [0]:
%run "./00_config"

# Core Functions – Metadata Driven JSON Loader

All functions live here. `02_main` just calls `run_pipeline_autoloader()`
(or `run_pipeline()` for the slower per-file path).

Run `%run "./00_config"` first (done automatically from `02_main`)

## 1. Small helper

In [0]:
def get_full_table_name(catalog, schema, table):
    return f"{catalog}.{schema}.{table}"

## 2. source_control table

The table you maintain by hand one row per source folder. Flip
`is_active` to 0 to pause a source without deleting it. Columns match
exactly what you gave us, nothing added.

| column | meaning |
|---|---|
| s_no | serial number |
| source | s3/adls/volume/dbfs |
| file_path | folder to scan (must end with `/`) |
| file_type | json |
| tgt_catalog / tgt_schema / tgt_table | where files from this source get loaded - all files under one source share one target table |
| is_active | 1 = pipeline processes this source, 0 = skipped |

In [0]:
def get_active_sources():
    ctl_full = get_full_table_name(meta_catalog, meta_schema, source_control_table)
    df = spark.table(ctl_full).filter(col("is_active") == 1)
    return [row.asDict() for row in df.collect()]

## 3. load_logs table

There's no separate file-status table. The log table itself is checked on
every run to work out which files already loaded successfully — a file's
**most recent** SUCCESS or FAILED log row is its current status.

| column | meaning |
|---|---|
| log_id | unique id for the log row |
| run_id | run id |
| executed_by | executor name |
| s_no | serial number |
| source / file_path / tgt_catalog / tgt_schema / tgt_table | target delta path|
| file_name | file name |
| status | SUCCESS / FAILED |
| start_time / end_time | timing |
| rows_processed | row count|
| message | detail, including error messages |

In [0]:
def write_log(status, source_row=None, file_name=None, rows_processed=None, message=None, start_time=None):

    now = datetime.now()
    sr = source_row or {}
    row = [(
        str(uuid.uuid4()), run_id, executed_by,
        sr.get("s_no"), sr.get("source"), sr.get("file_path"),
        sr.get("tgt_catalog"), sr.get("tgt_schema"), sr.get("tgt_table"),
        file_name, status, start_time or now, now, rows_processed, message,
    )]
    df_log = spark.createDataFrame(row, schema=LOG_SCHEMA)
    df_log.write.format("delta").mode("append") \
        .saveAsTable(get_full_table_name(meta_catalog, meta_schema, log_table))

## 4. File discovery and deciding what to process

`discover_files` lists every file currently sitting in a source's folder.

`get_files_to_process` then looks at the log table for that source, finds
each file's most recent SUCCESS/FAILED result, and returns:
- files that were never logged before (brand new files), plus
- files whose last attempt was FAILED

Files whose last attempt was SUCCESS are left out, so a re-run only
touches what's new or previously failed.

In [0]:
def discover_files(source_row):
    files = dbutils.fs.ls(source_row["file_path"])
    ext = "." + source_row["file_type"]
    return [f.name for f in files if f.name.endswith(ext)]


def get_latest_file_status(s_no):
    log_full = get_full_table_name(meta_catalog, meta_schema, log_table)
    df = (
        spark.table(log_full)
             .filter(col("s_no") == s_no)
             .filter(col("status").isin("SUCCESS", "FAILED"))
             .filter(col("file_name").isNotNull())
    )
    window = Window.partitionBy("file_name").orderBy(desc("end_time"))
    latest = (
        df.withColumn("rn", row_number().over(window))
          .filter(col("rn") == 1)
          .select("file_name", "status")
    )
    return {row.file_name: row.status for row in latest.collect()}


def get_files_to_process(source_row, discovered_files):
    latest_status = get_latest_file_status(source_row["s_no"])
    return [f for f in discovered_files if latest_status.get(f) != "SUCCESS"]

## 5. Loading a single file

Reads one file and appends it into the source's target table (all files
from the same source share one table, as set in source_control).

File names follow the pattern
`carereport_hsar_<id>_<period_start>_<period_end>_<dt_data_file>_<HHMMSS>`.
`dt_data_file` is the date the file was generated/extracted (the token
right before the trailing time-of-day) and is pulled out with
`extract_dt_data_file` below.

Target table columns:

| column | type | meaning |
|---|---|---|
| file_path | string | folder the file came from |
| file_name | string | file name |
| json_original_data | string | raw JSON file content |
| json_data_for_parsing | variant | same content, parsed as VARIANT |
| file_size_kb | double | file size in KB |
| file_modified_date | timestamp | file's last modified time |
| dt_data_file | string | date token extracted from the file name |
| dt_data_file_date | date | `dt_data_file` parsed as a date (yyyyMMdd) |
| data_year / data_month | int | year/month of `dt_data_file_date`, used to partition the target table |
| loaded_at | timestamp | when this row was loaded |

Both ingestion paths (`process_file` and the Autoloader path below) write
this exact same set of columns and partition on `data_year`/`data_month`,
so a source can be switched between the per-file and Autoloader entry
points without creating a schema mismatch on append.

In [0]:
def extract_dt_data_file(file_name):
    stem = file_name.rsplit(".", 1)[0]  # drop extension, if any
    parts = stem.split("_")
    return parts[-2] if len(parts) >= 2 else None

In [0]:
def process_file(source_row, file_name):
    file_type = source_row["file_type"].lower()
    file_full_path = source_row["file_path"] + file_name
    tgt_full = get_full_table_name(source_row["tgt_catalog"], source_row["tgt_schema"], source_row["tgt_table"])

    file_info = dbutils.fs.ls(file_full_path)[0]
    file_size_kb = file_info.size / 1024.0
    file_modified_date = datetime.fromtimestamp(file_info.modificationTime / 1000)
    dt_data_file = extract_dt_data_file(file_name)

    if file_type == "json":
        df_final = (
            spark.read
                 .option("wholetext", True)
                 .text(file_full_path)
                 .select(
                     lit(source_row["file_path"]).alias("file_path"),
                     lit(file_name).alias("file_name"),
                     col("value").alias("json_original_data"),
                     parse_json(col("value")).alias("json_data_for_parsing"),
                     lit(file_size_kb).alias("file_size_kb"),
                     lit(file_modified_date).alias("file_modified_date"),
                     lit(dt_data_file).alias("dt_data_file"),
                     to_date(lit(dt_data_file), "yyyyMMdd").alias("dt_data_file_date"),
                     current_timestamp().alias("loaded_at"),
                 )
                 .withColumn("data_year", year(col("dt_data_file_date")))
                 .withColumn("data_month", month(col("dt_data_file_date")))
        )
    else:
        raise NotImplementedError(f"file_type '{file_type}' is not supported yet")

    row_count = df_final.count()
    df_final.write.format("delta").mode("append") \
        .partitionBy("data_year", "data_month") \
        .saveAsTable(tgt_full)
    return row_count

## 5b. Autoloader ingestion (for large file counts)

`process_file` / `run_pipeline` above load one file per Spark job — fine
for a handful of files, far too slow once a source has tens of thousands
(each file pays for a separate text-read job, a `.count()` job, and a log
write, all sequential).

`run_pipeline_autoloader` uses Databricks Autoloader (`cloudFiles`) instead:
- file discovery is incremental and checkpointed, not a full listing every run
- files are read and written in micro-batches, not one Spark job per file
- logging happens once per micro-batch, not once per file
- no schema extraction — just the raw text, the parsed VARIANT, and file
  metadata, matching the target table columns above

Uses `trigger(availableNow=True)`: it processes everything currently sitting
in the folder as a series of batches and then stops, rather than running as
a continuous stream.

In [0]:
def get_checkpoint_path(source_row):
    base = source_row["tgt_checkpoint_path"].rstrip("/")
    return f"{base}/checkpoint"


def get_schema_path(source_row):
    base = source_row["tgt_checkpoint_path"].rstrip("/")
    return f"{base}/schema"


def _write_autoloader_batch(source_row, tgt_full):
    """Return a foreachBatch function bound to this source's target table."""
    def _inner(batch_df, batch_id):
        if batch_df.isEmpty():
            return

        batch_df = batch_df.cache()
        batch_df.write.format("delta").mode("append")\
        .partitionBy("data_year", "data_month")\
        .saveAsTable(tgt_full)

        now = datetime.now()
        log_df = (
            batch_df.select("file_name")
                    .withColumn("log_id", expr("uuid()"))
                    .withColumn("run_id", lit(run_id))
                    .withColumn("executed_by", lit(executed_by))
                    .withColumn("s_no", lit(source_row["s_no"]))
                    .withColumn("source", lit(source_row["source"]))
                    .withColumn("file_path", lit(source_row["file_path"]))
                    .withColumn("tgt_catalog", lit(source_row["tgt_catalog"]))
                    .withColumn("tgt_schema", lit(source_row["tgt_schema"]))
                    .withColumn("tgt_table", lit(source_row["tgt_table"]))
                    .withColumn("status", lit("SUCCESS"))
                    .withColumn("start_time", lit(now))
                    .withColumn("end_time", lit(now))
                    .withColumn("rows_processed", lit(1).cast(LongType()))
                    .withColumn("message", lit("Loaded via Autoloader"))
                    .select(
                        "log_id", "run_id", "executed_by", "s_no", "source", "file_path",
                        "tgt_catalog", "tgt_schema", "tgt_table", "file_name", "status",
                        "start_time", "end_time", "rows_processed", "message",
                    )
        )
        log_df.write.format("delta").mode("append") \
            .saveAsTable(get_full_table_name(meta_catalog, meta_schema, log_table))

        batch_df.unpersist()

    return _inner


def process_source_autoloader(source_row):
    
    file_type = source_row["file_type"].lower()
    if file_type != "json":
        raise NotImplementedError(f"file_type '{file_type}' is not supported yet")

    tgt_full = get_full_table_name(source_row["tgt_catalog"], source_row["tgt_schema"], source_row["tgt_table"])

    raw_stream = (
        spark.readStream.format("cloudFiles")
             .option("cloudFiles.format", "text")
             .option("wholetext", "true")
             .option("cloudFiles.schemaLocation", get_schema_path(source_row))
             .option("cloudFiles.includeExistingFiles", "true")
             .option("cloudFiles.allowOverwrites", "true")
             .load(source_row["file_path"])
    )


    # file-name and file_Date extraction: carereport_hsar_<id>_20220101_20251106_20251113_102927.json -> 20251113
    name_no_ext = regexp_extract(col("_metadata.file_name"), r"^(.*?)(\.[^.]+)?$", 1)
    dt_data_file_expr = element_at(split(name_no_ext, "_"), -2)
    file_date_expr = to_date(dt_data_file_expr, "yyyyMMdd")

    transformed = raw_stream.select(
        lit(source_row["file_path"]).alias("file_path"),
        col("_metadata.file_name").alias("file_name"),
        col("value").alias("json_original_data"),
        parse_json(col("value")).alias("json_data_for_parsing"),
        (col("_metadata.file_size") / 1024.0).alias("file_size_kb"),
        col("_metadata.file_modification_time").alias("file_modified_date"),
        dt_data_file_expr.alias("dt_data_file"),
        file_date_expr.alias("dt_data_file_date"),
        year(file_date_expr).alias("data_year"),
        month(file_date_expr).alias("data_month"),
        current_timestamp().alias("loaded_at"),
    )

    query = (
        transformed.writeStream
                   .foreachBatch(_write_autoloader_batch(source_row, tgt_full))
                   .option("checkpointLocation", get_checkpoint_path(source_row))
                   .trigger(availableNow=True)
                   .start()
    )
    query.awaitTermination()
    return query.lastProgress


def run_pipeline_autoloader():
    
    create_log_table()

    print("Autoloader pipeline run started")
    active_sources = get_active_sources()
    print(f"{len(active_sources)} active source(s) found in source_control")

    for source_row in active_sources:
        print(f"Source {source_row['s_no']}: starting Autoloader ingestion")
        process_source_autoloader(source_row)
        print(f"Source {source_row['s_no']}: Autoloader batch complete")

    print("Autoloader pipeline finished")

## 6. Orchestration (per-file path)

The slower entry point: for every active source, list all files in the
folder, work out which ones still need loading (new or previously
failed), and load them one by one — one file failing doesn't stop the
others, and every outcome is logged. Use `run_pipeline_autoloader()`
instead for large file counts.

In [0]:
def run_pipeline():

    create_log_table()

    print("Pipeline run started")

    active_sources = get_active_sources()
    print(f"{len(active_sources)} active source(s) found in source_control")

    total_success = 0
    total_fail = 0

    for source_row in active_sources:
        discovered = discover_files(source_row)
        print(f"Source {source_row['s_no']}: found {len(discovered)} file(s) in folder")

        files_to_process = get_files_to_process(source_row, discovered)
        print(f"Source {source_row['s_no']}: {len(files_to_process)} file(s) are new or previously failed, will be processed")

        for file_name in files_to_process:
            start_time = datetime.now()
            try:
                rows = process_file(source_row, file_name)
                write_log(
                    status="SUCCESS", source_row=source_row, file_name=file_name,
                    rows_processed=rows, start_time=start_time,
                    message="File loaded successfully",
                )
                total_success += 1
            except Exception as e:
                error_message = str(e)[:500]
                write_log(
                    status="FAILED", source_row=source_row, file_name=file_name,
                    start_time=start_time, message=error_message,
                )
                total_fail += 1

    print(f"Pipeline finished. Success: {total_success}, Failed: {total_fail}")


## 7. File/Record Versioning

The `create_versioned_table()` function creates a versioned copy of each active target table.

- It retrieves active source configurations using `get_active_sources()`.
- It constructs each target table name using the control-table fields:
  `tgt_catalog`, `tgt_schema`, and `tgt_table`.
- It partitions records using `versioning_partition_field` (for example, `file_name`).
- It assigns a sequential `file_version` for each partition, ordered by `file_modified_date`.
- The versioned data is saved as `<target_table>_versioned` in overwrite mode.

Example:

`iics.casemix.json_test` → `iics.casemix.json_test_versioned`

In [0]:
def create_versioned_table():
    control_rows = get_active_sources()

    for row in control_rows:
        target_table = (
            f"{row['tgt_catalog']}."
            f"{row['tgt_schema']}."
            f"{row['tgt_table']}"
        )
        partition_field = row["versioning_partition_field"]

        df = spark.sql(f"""
            SELECT
                ROW_NUMBER() OVER (
                    PARTITION BY {partition_field}
                    ORDER BY file_modified_date
                ) AS file_version,
                *
            FROM {target_table}
        """)

        (
            df.write
              .mode("overwrite")
              .saveAsTable(f"{target_table}_versioned")
        )

        print(f"Created: {target_table}_versioned")